# Type1 Prediction API with Kaggle + ngrok

Run this notebook top-to-bottom on Kaggle. It loads the model once, starts a FastAPI server, exposes it with ngrok, and accepts one query per `POST /predict`. Requests are serialized with a lock so GPU inference runs sequentially.

## 0. Install dependencies

Kaggle usually needs ngrok/FastAPI installed in the session. Enable Internet for this notebook and set `NGROK_AUTHTOKEN` in Kaggle secrets or `os.environ` before starting the server.

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio transformers accelerate bitsandbytes z3-solver


In [ ]:
import os
import re
import json
import time
import threading
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
import z3
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from fastapi import FastAPI
from fastapi.responses import JSONResponse
import uvicorn
import nest_asyncio
from pyngrok import ngrok

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)

@dataclass
class CFG:
    # Downloads directly from Hugging Face. Kaggle Internet must be ON.
    # Optional: add HF_TOKEN as a Kaggle Secret if Hugging Face rate limits or auth is required.
    model_qwen_path: str = os.environ.get("TYPE1_MODEL_NAME", "Qwen/Qwen2.5-Coder-7B-Instruct")
    qwen_4bit: bool = True
    max_new_classify: int = 64
    max_new_convert: int = 1024
    max_new_explain: int = 256
    z3_timeout_ms: int = 5000
    api_host: str = "0.0.0.0"
    api_port: int = 8000
    ngrok_authtoken: str = os.environ.get("NGROK_AUTHTOKEN", "")

cfg = CFG()
print("Type 1 model:", cfg.model_qwen_path)


## 1. Load Qwen model

In [ ]:
def load_causal(model_path, four_bit=False):
    hf_token = os.environ.get("HF_TOKEN", "").strip() or None

    print("Loading Type 1 model from Hugging Face:", model_path)
    print("local_files_only: False")

    tok = AutoTokenizer.from_pretrained(
        model_path,
        local_files_only=False,
        token=hf_token,
        trust_remote_code=True,
    )

    kw = dict(
        device_map="auto",
        torch_dtype=torch.float16,
        local_files_only=False,
        token=hf_token,
        trust_remote_code=True,
    )

    if four_bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(model_path, **kw)
    model.eval()
    return tok, model

qwen_tok, qwen = load_causal(cfg.model_qwen_path, four_bit=cfg.qwen_4bit)
print("Qwen loaded for API inference.")

@torch.inference_mode()
def chat(tok, model, system, user, max_new):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": user})
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **ids,
        max_new_tokens=max_new,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True).strip()


## 2. Pipeline components

In [ ]:
_OPT = re.compile(r'^\s*([A-D])[\.\)]\s*(.+)$')
_LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def _normalize_options(options):
    if not options:
        return {}
    choices = {}
    if isinstance(options, dict):
        for key, value in options.items():
            label = str(key).strip().upper()[:1]
            if label:
                choices[label] = str(value).strip()
        return choices
    if isinstance(options, list):
        for i, item in enumerate(options):
            default_label = _LETTERS[i] if i < len(_LETTERS) else str(i + 1)
            label = default_label
            text = item
            if isinstance(item, dict):
                label = str(item.get("label", item.get("key", default_label))).strip().upper()[:1]
                text = item.get("text", item.get("option", item.get("value", item.get("answer", ""))))
            else:
                m = _OPT.match(str(item))
                if m:
                    label, text = m.group(1), m.group(2)
            if label and str(text).strip():
                choices[label] = str(text).strip()
    return choices

def parse_choices(question, options=None):
    """Return (stem, {A: text, ...}). Prefer explicit API options when present."""
    explicit = _normalize_options(options)
    lines = str(question or "").split("\n")
    stem, parsed = [], {}
    for ln in lines:
        m = _OPT.match(ln)
        if m:
            parsed[m.group(1)] = m.group(2).strip()
        else:
            stem.append(ln)
    return " ".join(stem).strip(), (explicit or parsed)

CLASSIFY_SYS = (
    "You classify an educational logic question. "
    "Reply with ONE word only: 'mcq' if it lists labelled options A-D, else 'yesno'."
)

def classify(question, options=None):
    stem, choices = parse_choices(question, options=options)
    if len(choices) >= 2:
        return {"question_type": "mcq", "normalized_question": stem, "choices": choices}
    verdict = chat(qwen_tok, qwen, CLASSIFY_SYS, str(question), cfg.max_new_classify).lower()
    qtype = "mcq" if "mcq" in verdict else "yesno"
    return {"question_type": qtype, "normalized_question": stem or str(question), "choices": choices}


In [ ]:
CONVERT_SYS = """You convert natural-language logic statements into parser-safe First-Order Logic.

  Goal:
  - Convert all input lines into one shared FOL vocabulary.
  - The dataset usually has an implicit object domain: repeated subject categories are not logical conditions.
  - Reuse the same predicate name for the same property/relation across premises and query.
  - Do not merge concepts just because they are related, causal, or commonly associated.
  - Never return an empty string for a simple factual, universal, existential, or implication statement.

  Hard rules:
  - Output parser-safe FOL strings using only:
    ForAll(var, ...)
    Exists(var, ...)
    Not(...)
    Implies(A, B)
    And(A, B)
    Or(A, B)
    Predicate(arg1, arg2, ...)
  - Variables may be x, y, s, c, m, d, etc.; use nested quantifiers for multiple variables.
  - Predicates and constants must be ASCII identifiers only: CamelCase/PascalCase or snake_case. No spaces, quotes, dots, hyphens, or punctuation inside identifiers.
  - Multi-argument predicates are allowed when the relation needs objects/constants, e.g. HasDegree(John, PhD), Enroll(x, CourseB), Higher(PhD, MSc).
  - Do not output arithmetic/comparison syntax such as >, <, >=, <=, =, +, *, decimals, or percentages. Predicateize thresholds instead, e.g. GPA above 3.5 -> GpaAbove3_5(x), attendance less than 50% -> AttendanceBelow50(x).
  - Never output True, False, BoolVal, tautologies, x = x, or placeholder predicates.

  Dynamic domain/category rules - VERY IMPORTANT:
  - Infer the base domain noun phrase from the current record, e.g. drones, students, cloud services, autonomous vehicles, hardware components, Python projects.
  - The base domain is implicit. Do NOT require it to appear in a fixed list.
  - Do NOT create a predicate just for the base domain unless the statement is explicitly about membership in that category.
  - Always extract the property/relation asserted about the domain, even if the domain noun is new.
  - "All/Every/Any <base domain> P" -> ForAll(x, P(x)), NOT ForAll(x, Implies(Domain(x), P(x))).
  - "All drones function correctly" -> ForAll(x, FunctionCorrectly(x)).
  - "All cloud services are cost-effective" -> ForAll(x, CostEffective(x)).
  - "Every safe vehicle is thoroughly tested" -> ForAll(x, Implies(Safe(x), ThoroughlyTested(x))). Here Vehicle is domain, Safe is a condition.
  - "There exists at least one <base domain> that is P" -> Exists(x, P(x)), NOT Exists(x, And(Domain(x), P(x))).
  - "If a/an <base domain> is A, then it is B" -> ForAll(x, Implies(A(x), B(x))).
  - "If a/an <base domain> does not A, then it does not B" -> ForAll(x, Implies(Not(A(x)), Not(B(x)))).

  Vocabulary rules:
  - Build one canonical predicate inventory before writing the final answers.
  - Reuse identical spelling for the same property/relation.
  - Keep different properties separate when in doubt.
  - Collapse only trivial wording variants, spelling variants, abbreviations, or direct paraphrases.
  - Negated forms must use Not(same predicate), e.g. "not well-tested" -> Not(WellTested(x)).
  - Ignore weak modifiers such as very, highly, generally, usually, often, likely, may, can, tends to.
  - Do NOT ignore "not necessarily". In this dataset it is a negation cue; convert it with Not(...) and never as a tautology.

  Semantic rules:
  - Preserve quantifiers and logical meaning exactly after removing implicit base-domain nouns.
  - Preserve implication direction. Never reverse antecedent and consequent.
  - Object-level rule: "If A, then B" -> ForAll(x, Implies(A(x), B(x))) when A/B are properties of the same implicit object.
  - Relative-clause rule: "<domain> who/that A and B are C" -> ForAll(x, Implies(And(A(x), B(x)), C(x))).
  - Sentence-level rule: "If there exists at least one A, then any/every B is C" -> Implies(Exists(x, A(x)), ForAll(x, Implies(B(x), C(x)))).
  - Sentence-level rule: "If there exists at least one A, then there exists at least one B" -> Implies(Exists(x, A(x)), Exists(x, B(x))).
  - Sentence-level rule: "If all/every A, then B implies C" -> Implies(ForAll(x, A(x)), ForAll(x, Implies(B(x), C(x)))).
  - Do NOT collapse a sentence-level existential or universal antecedent into ForAll(x, Implies(A(x), B(x))).
  - Necessary/required direction: "A is necessary/required to B" means B -> A.
  - Modal verbs such as must/can/eligible/qualifies usually express implication from condition to capability/requirement.
  - "Either A or B" and "A or B" -> Or(A, B).
  - "P unless Q" -> Or(P, Q) when it appears as a consequent; otherwise Implies(Not(Q), P).
  - Named facts become predicate applications over constants, e.g. Sophia completed the core curriculum -> CompletedCoreCurriculum(Sophia).
  - Universal questions like "Are all/every <domain> P?" -> ForAll(x, P(x)); never use Implies(P(x), True).
  - Do not add extra facts not present in the text.

  Output rules:
  - Output ONLY a JSON list of strings.
  - One FOL string per input line.
  - No commentary, no explanation, no markdown.
  """

FEWSHOT = """Input premises:
1. All drones function correctly.
2. All cloud services are cost-effective.
3. Every safe vehicle is thoroughly tested.
4. There exists at least one Python project that has clean and readable code.
5. If a Python project is optimized, then it has clean and readable code.
6. If a Python code does not follow PEP8, then it is not well-tested.
7. Students who have completed the core curriculum and passed the science assessment are qualified for advanced courses.
8. Sophia has completed the core curriculum.
9. If there exists at least one energy-efficient autonomous vehicle, then any safe vehicle is thoroughly tested.
10. If there exists at least one secure cloud service, then there exists at least one cost-effective service.
11. If all microchips pass quality tests, then if a microchip does not have a stable power supply, it will not operate efficiently.
12. MSc is higher than BA.
13. Dr John has a PhD.
14. If a student completes Course A, they can enroll in Course B.
15. Mastery of quantum superposition is necessary to comprehend quantum measurement.
16. If a student masters the course content, they either pass a proctored exam or submit a capstone project.
17. If a student does not master the course content and submits a capstone, the capstone is rejected unless re-enrolled.
18. There does not necessarily exist an autonomous vehicle that is thoroughly tested.
19. If a student maintains a GPA above 3.5, they graduate with honors.

Output:
["ForAll(x, FunctionCorrectly(x))",
"ForAll(x, CostEffective(x))",
"ForAll(x, Implies(Safe(x), ThoroughlyTested(x)))",
"Exists(x, CleanReadable(x))",
"ForAll(x, Implies(Optimized(x), CleanReadable(x)))",
"ForAll(x, Implies(Not(PEP8(x)), Not(WellTested(x))))",
"ForAll(x, Implies(And(CompletedCoreCurriculum(x), PassedScienceAssessment(x)), QualifiedAdvancedCourses(x)))",
"CompletedCoreCurriculum(Sophia)",
"Implies(Exists(x, EnergyEfficient(x)), ForAll(y, Implies(Safe(y), ThoroughlyTested(y))))",
"Implies(Exists(x, Secure(x)), Exists(y, CostEffective(y)))",
"Implies(ForAll(x, PassesQualityTests(x)), ForAll(y, Implies(Not(StablePowerSupply(y)), Not(OperateEfficiently(y)))))",
"Higher(MSc, BA)",
"HasDegree(DrJohn, PhD)",
"ForAll(x, Implies(Complete(x, CourseA), Enroll(x, CourseB)))",
"ForAll(x, Implies(ComprehendQuantumMeasurement(x), MasterQuantumSuperposition(x)))",
"ForAll(x, Implies(MasterCourseContent(x), Or(PassProctoredExam(x), SubmitCapstoneProject(x))))",
"ForAll(x, Implies(And(Not(MasterCourseContent(x)), SubmitCapstoneProject(x)), Or(CapstoneRejected(x), ReEnrolled(x))))",
"Not(Exists(x, ThoroughlyTested(x)))",
"ForAll(x, Implies(GpaAbove3_5(x), GraduateWithHonors(x)))"]"""

def convert_premises(premises):
    body = "\n".join(f"{i+1}. {p}" for i, p in enumerate(premises))
    user = f"{FEWSHOT}\n\nInput premises:\n{body}\nOutput:"
    raw = chat(qwen_tok, qwen, CONVERT_SYS, user, cfg.max_new_convert)
    return _safe_json_list(raw, n=len(premises))

def _predicate_inventory_from_fol(fol_strings):
    names = []
    for fol in fol_strings or []:
        if not fol:
            continue
        for name in re.findall(r"\b([A-Z][A-Za-z0-9_]*)\s*\(", fol):
            if name not in {"ForAll", "Exists", "Not", "Implies", "And", "Or"} and name not in names:
                names.append(name)
    return names

def _format_converted_premises(premises, fol_premises):
    lines = []
    for i, nl in enumerate(premises, start=1):
        fol = fol_premises[i-1] if fol_premises and i-1 < len(fol_premises) else ""
        lines.append(f"{i}. NL: {nl}\n   FOL: {fol}")
    return "\n".join(lines)

def convert_query(stem, premises, fol_premises=None):
    converted_context = _format_converted_premises(premises, fol_premises)
    inventory = _predicate_inventory_from_fol(fol_premises)
    inventory_text = "\n".join(f"- {name}" for name in inventory) if inventory else "- No predicates available"
    user = (
        f"{FEWSHOT}\n\n"
        "Use the same implicit-domain and predicate rules for this new dataset.\n"
        "The premises below have ALREADY been converted. Reuse their predicate names exactly.\n"
        "Do not invent a synonym predicate when an existing predicate has the same meaning.\n\n"
        f"Converted premises for vocabulary context:\n{converted_context}\n\n"
        f"Predicate inventory already used:\n{inventory_text}\n\n"
        "Convert the following natural-language question as the proposition being tested.\n"
        "Question conversion rules:\n"
        "- Yes/no question 'Are all/every <domain> P?' -> ForAll(x, P(x)); never Implies(P(x), True).\n"
        "- Yes/no question 'Is there at least one <domain> that is P?' -> Exists(x, P(x)).\n"
        "- 'Does it follow that ...?' means convert only the embedded claim.\n"
        "- If the embedded claim has 'if there exists ..., then any/every ...', preserve it as Implies(Exists(...), ForAll(...)).\n"
        "- Never output True, False, BoolVal, or tautologies.\n"
        "- Ignore implicit domain nouns exactly as in premise conversion.\n"
        "- Reuse predicates from the inventory whenever semantically equivalent.\n"
        "- Return ONLY a JSON list with exactly one FOL string.\n\n"
        f"Question target:\n{stem}\nOutput:"
    )
    out = _safe_json_list(chat(qwen_tok, qwen, CONVERT_SYS, user, 256), n=1)
    return out[0] if out else None

def _normalize_fol_candidate(fol):
    if fol is None:
        return None
    fol = str(fol).strip().strip('`')
    fol = re.sub(r"^json\s*", "", fol, flags=re.I).strip()
    fol = re.sub(r"^```(?:json)?", "", fol, flags=re.I).strip()
    fol = re.sub(r"```$", "", fol).strip()

    # Common bad query conversion: Are all P? -> ForAll(x, Implies(P(x), True)).
    m = re.fullmatch(r"ForAll\(x,\s*Implies\(([^,]+\(x\)),\s*True\)\s*\)", fol)
    if m:
        return f"ForAll(x, {m.group(1)})"

    # True/False are outside the supported fragment for this pipeline.
    if re.search(r"\b(True|False|BoolVal)\b", fol):
        return None
    return fol

def _safe_json_list(raw, n):
    m = re.search(r"\[.*\]", raw, re.S)
    if m:
        try:
            lst = json.loads(m.group(0))
            lst = [_normalize_fol_candidate(x) for x in lst]
        except Exception:
            lst = []
    else:
        # Query conversion sometimes returns a bare FOL string. Keep it instead of dropping to None.
        candidate = _normalize_fol_candidate(raw)
        lst = [candidate] if candidate and re.match(r"^(ForAll|Exists|Not|Implies|And|Or|[A-Z][A-Za-z0-9_]*\()", candidate) else []
    if len(lst) < n: lst += [None]*(n-len(lst))
    return lst[:n]

In [ ]:
class FOLParser:
    QUANT = {"∀": "ForAll", "∃": "Exists"}
    def __init__(self):
        self.Obj = z3.DeclareSort("Obj")
        self.preds = {}            # name -> z3 Function
        self.scope = {}            # var name -> z3 Const

    def pred(self, name, arity=1):
        if name not in self.preds:
            self.preds[name] = z3.Function(name, *([self.Obj]*arity), z3.BoolSort())
        return self.preds[name]

    def prop(self, name):
        if name not in self.preds:
            self.preds[name] = z3.Bool(name)        # 0-ary propositional atom
        return self.preds[name]

    # ---- tokenizer ----
    def tokenize(self, s):
        s = (s.replace("->", "→").replace("<->", "↔").replace("~", "¬")
               .replace("&", "∧").replace("|", "∨"))
        toks, i = [], 0
        spec = {"(": "LP", ")": "RP", ",": "CM", "¬": "NOT",
                "∧": "AND", "∨": "OR", "→": "IMP", "↔": "IFF"}
        while i < len(s):
            c = s[i]
            if c.isspace(): i += 1; continue
            if c in ("∀", "∃"): toks.append(("Q", self.QUANT[c])); i += 1; continue
            if c in spec: toks.append((spec[c], c)); i += 1; continue
            if c.isalnum() or c == "_":
                j = i
                while j < len(s) and (s[j].isalnum() or s[j] == "_"): j += 1
                w = s[i:j]; i = j; wl = w.lower()
                if wl in ("forall", "exists"):
                    toks.append(("Q", "ForAll" if wl == "forall" else "Exists"))
                elif wl == "not": toks.append(("NOT", w))
                elif wl == "and": toks.append(("AND2", w))
                elif wl == "or":  toks.append(("OR2", w))
                elif wl == "implies": toks.append(("IMP2", w))
                else: toks.append(("ID", w))
                continue
            i += 1                                   # skip unknown char
        toks.append(("END", None)); return toks

    # ---- recursive descent (precedence: IFF < IMP < OR < AND < NOT) ----
    def parse(self, s):
        self.toks = self.tokenize(s); self.p = 0
        expr = self._iff()
        if self._peek()[0] != "END":
            raise ValueError(f"trailing token after complete formula: {self._peek()}")
        return expr
    def _peek(self): return self.toks[self.p]
    def _eat(self, t=None):
        tok = self.toks[self.p]
        if t and tok[0] != t: raise ValueError(f"expected {t}, got {tok}")
        self.p += 1; return tok
    def _iff(self):
        a = self._imp()
        while self._peek()[0] == "IFF": self._eat(); a = (a == self._imp())
        return a
    def _imp(self):
        a = self._or()
        if self._peek()[0] == "IMP":
            self._eat(); return z3.Implies(a, self._imp())
        return a
    def _or(self):
        a = self._and()
        while self._peek()[0] == "OR": self._eat(); a = z3.Or(a, self._and())
        return a
    def _and(self):
        a = self._not()
        while self._peek()[0] == "AND": self._eat(); a = z3.And(a, self._not())
        return a
    def _not(self):
        if self._peek()[0] == "NOT":
            self._eat(); return z3.Not(self._not())
        return self._atom()
    def _atom(self):
        kind, val = self._peek()
        if kind == "Q":                                   # quantifier
            self._eat()
            if self._peek()[0] == "LP":                   # ForAll(x, body)
                self._eat("LP"); var = self._eat("ID")[1]
                self._eat("CM"); v = z3.Const(var, self.Obj)
                self.scope[var] = v; body = self._iff(); self._eat("RP")
            else:                                         # ∀x (body)
                var = self._eat("ID")[1]; v = z3.Const(var, self.Obj)
                self.scope[var] = v; self._eat("LP"); body = self._iff(); self._eat("RP")
            return z3.ForAll(v, body) if val == "ForAll" else z3.Exists(v, body)
        if kind in ("AND2", "OR2", "IMP2"):               # And(A,B)/Or(A,B)/Implies(A,B)
            self._eat(); self._eat("LP"); a = self._iff(); self._eat("CM"); b = self._iff(); self._eat("RP")
            return {"And2": z3.And, "Or2": z3.Or, "Implies": z3.Implies}[
                {"AND2":"And2","OR2":"Or2","IMP2":"Implies"}[kind]](a, b)
        if kind == "NOT":
            self._eat(); self._eat("LP"); a = self._iff(); self._eat("RP"); return z3.Not(a)
        if kind == "LP":
            self._eat(); a = self._iff(); self._eat("RP"); return a
        if kind == "ID":                                  # predicate Name(arg) OR bare prop
            name = self._eat("ID")[1]
            if self._peek()[0] != "LP":                   # propositional atom: P, Q, R...
                return self.prop(name)
            self._eat("LP")
            args = [self._eat("ID")[1]]
            while self._peek()[0] == "CM": self._eat(); args.append(self._eat("ID")[1])
            self._eat("RP")
            zargs = [self.scope.get(a, z3.Const(a, self.Obj)) for a in args]
            return self.pred(name, len(zargs))(*zargs)
        raise ValueError(f"unexpected token {kind}")


In [ ]:
def _build(premises_z3):
    parser = FOLParser()
    return parser

def entails_with_core(premises_z3, query, timeout=cfg.z3_timeout_ms):
    """Return (entailed: bool, core_idx_0based: list)."""
    if query is None: return False, []
    s = z3.Solver(); s.set("timeout", timeout); s.set(unsat_core=True)
    trackers = []
    for i, p in enumerate(premises_z3):
        if p is None: continue
        t = z3.Bool(f"p{i}"); s.assert_and_track(p, t); trackers.append((t, i))
    s.add(z3.Not(query))
    if s.check() == z3.unsat:
        core = set(map(str, s.unsat_core()))
        idx = sorted(i for (t, i) in trackers if str(t) in core)
        return True, idx
    return False, []

def reason_yesno(premises_z3, query):
    ok, core = entails_with_core(premises_z3, query)
    if ok: return "Yes", core, "UNSAT(P∧¬Q)"
    ok, core = entails_with_core(premises_z3, z3.Not(query)) if query is not None else (False, [])
    if ok: return "No", core, "UNSAT(P∧Q)"
    return "Unknown", [], "SAT both ways"

def reason_mcq(premises_z3, option_queries):
    """option_queries: dict {A: z3expr,...}. Return (letter|None, core, trace)."""
    winners = []
    for letter, q in option_queries.items():
        ok, core = entails_with_core(premises_z3, q)
        if ok: winners.append((letter, core))
    if not winners: return None, [], "no option entailed"
    letter, core = min(winners, key=lambda lc: len(lc[1]) or 99)   # fewest premises
    return letter, core, f"UNSAT, {len(winners)} option(s) entailed"

In [ ]:
def _safe_json_object(raw):
    m = re.search(r"\{.*\}", str(raw), re.S)
    if not m:
        return {}
    try:
        return json.loads(m.group(0))
    except Exception:
        return {}

def _normalize_premise_idx(values, n_premises, max_k=4):
    idx = []
    for value in values or []:
        if isinstance(value, int) or str(value).isdigit():
            i = int(value)
            if 1 <= i <= n_premises:
                idx.append(i)
    return sorted(dict.fromkeys(idx))[:max_k]

SUPPORTED_EXPLAIN_SYS = (
    "You are a logic tutor. Answer in 1-2 concise sentences. "
    "Explain only why the answer follows from the cited premises. "
    "Cite premise numbers and avoid extra background."
)

UNCERTAIN_EXPLAIN_SYS = (
    "You are a logic tutor for formal reasoning. The system verdict is Unknown. "
    "Select only the premise numbers directly relevant to the attempted conclusion, then write a brief explanation. "
    "Return ONLY valid JSON with keys: premises_used and explanation. "
    "premises_used must be a JSON list of up to 6 premise numbers. "
    "The explanation must be 2-4 concise sentences: summarize what the selected premises establish and what link is missing. "
    "Do not reveal hidden chain-of-thought, invent facts, or choose a definite Yes/No/option answer."
)

def _format_premises(premises_nl, idx_1based):
    return "\n".join(
        f"Premise {i}: {premises_nl[i-1]}"
        for i in idx_1based
        if 1 <= i <= len(premises_nl)
    )

def _format_all_premises(premises_nl):
    return "\n".join(f"{i}. {p}" for i, p in enumerate(premises_nl, start=1))

def _make_supported_explanation_prompt(answer, premises_nl, idx_1based, question):
    cited = _format_premises(premises_nl, idx_1based)
    user = (
        f"Question: {question}\n"
        f"Answer: {answer}\n"
        f"Supporting premises:\n{cited}\n\n"
        "Explanation:"
    )
    return {"system": SUPPORTED_EXPLAIN_SYS, "user": user}

def _make_unknown_explanation_prompt(premises_nl, question, max_k=4):
    user = (
        f"Question:\n{question}\n\n"
        f"Premises:\n{_format_all_premises(premises_nl)}\n\n"
        f"Return JSON only. Use up to {max_k} directly relevant premise numbers."
    )
    return {"system": UNCERTAIN_EXPLAIN_SYS, "user": user}

def generate_supported_explanation(answer, premises_nl, idx_1based, question):
    prompt = _make_supported_explanation_prompt(answer, premises_nl, idx_1based, question)
    return chat(qwen_tok, qwen, prompt["system"], prompt["user"], cfg.max_new_explain)

def generate_unknown_result(premises_nl, question, max_k=4):
    prompt = _make_unknown_explanation_prompt(premises_nl, question, max_k=max_k)
    raw = chat(qwen_tok, qwen, prompt["system"], prompt["user"], cfg.max_new_explain)
    obj = _safe_json_object(raw)
    idx = _normalize_premise_idx(obj.get("premises_used", []), len(premises_nl), max_k=max_k)
    explanation = str(obj.get("explanation", "")).strip()
    if not explanation:
        explanation = str(raw).strip()
    return idx, explanation

def generate_explanation(answer, premises_nl, idx_1based, question):
    return generate_supported_explanation(answer, premises_nl, idx_1based, question)

## 3. Single-query prediction function

In [ ]:
def normalize_api_payload(payload: Dict[str, Any]) -> Dict[str, Any]:
    premises = payload.get("premises", payload.get("premises-NL", payload.get("premises_nl")))
    question = payload.get("query", payload.get("question"))
    if premises is None or question is None:
        raise ValueError("payload must contain premises and query/question")
    if not isinstance(premises, list):
        raise ValueError("premises must be a list of natural-language strings")
    return {
        "query_id": str(payload.get("query_id", payload.get("id", ""))),
        "type": payload.get("type", "type1"),
        "premises": [str(x) for x in premises],
        "question": str(question),
        "options": payload.get("options", []),
    }

def _parse_fol_list(parser, premises_nl, fol_premises):
    premises_z3 = []
    for fol in fol_premises:
        try:
            premises_z3.append(parser.parse(fol) if fol else None)
        except Exception:
            premises_z3.append(None)
    return premises_z3

def run_pipeline(sample: Dict[str, Any], verbose: bool = False) -> Dict[str, Any]:
    query_id = str(sample.get("query_id", ""))
    premises_nl = sample["premises"]
    question = sample["question"]
    options = sample.get("options", [])

    cls = classify(question, options=options)
    qtype, stem, choices = cls["question_type"], cls["normalized_question"], cls["choices"]

    parser = FOLParser()
    fol_premises = convert_premises(premises_nl)
    premises_z3 = _parse_fol_list(parser, premises_nl, fol_premises)

    if qtype == "mcq":
        opt_q = {}
        for label, text in choices.items():
            fol = convert_query(text, premises_nl, fol_premises)
            try:
                opt_q[label] = parser.parse(fol) if fol else None
            except Exception:
                opt_q[label] = None
        letter, core, trace = reason_mcq(premises_z3, {k: v for k, v in opt_q.items() if v is not None})
        if letter is not None:
            answer, idx = letter, sorted(i + 1 for i in core)
        else:
            answer, idx = "Unknown", []
    else:
        fol_q = convert_query(stem, premises_nl, fol_premises)
        try:
            query = parser.parse(fol_q) if fol_q else None
        except Exception:
            query = None
        answer, core, trace = reason_yesno(premises_z3, query)
        idx = [] if answer == "Unknown" else sorted(i + 1 for i in core)

    if str(answer).strip().lower() == "unknown":
        idx, explanation = generate_unknown_result(premises_nl, stem)
    else:
        explanation = generate_explanation(answer, premises_nl, idx, stem)

    return {
        "query_id": query_id,
        "answer": answer,
        "unit": "",
        "explanation": explanation,
        "premises_used": idx,
        "reasoning": "",
    }

def error_prediction(payload: Dict[str, Any], exc: Exception) -> Dict[str, Any]:
    return {
        "query_id": str(payload.get("query_id", payload.get("id", ""))),
        "answer": "Unknown",
        "unit": "",
        "explanation": f"Pipeline error: {type(exc).__name__}: {exc}",
        "premises_used": [],
        "reasoning": "",
    }


## 4. FastAPI app

Input JSON supports the official fields: `query_id`, `type`, `query`, `premises`, `options`. The endpoint returns exactly one prediction JSON with `query_id`, `answer`, `unit`, `explanation`, `premises_used`, and `reasoning`.

In [ ]:
app = FastAPI(title="Logic-Based Educational Queries Prediction API")
REQUEST_LOCK = threading.Lock()

@app.get("/")
def root():
    return {"status": "ok", "endpoint": "/predict", "method": "POST"}

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": qwen is not None}

@app.post("/predict")
def predict(payload: Dict[str, Any]):
    # Serialize requests explicitly. Kaggle GPU inference should process one query at a time.
    with REQUEST_LOCK:
        try:
            sample = normalize_api_payload(payload)
            return run_pipeline(sample, verbose=False)
        except Exception as exc:
            return JSONResponse(status_code=200, content=error_prediction(payload, exc))


## 5. Start server and publish with ngrok

Before running this cell, set the token if needed:

```python
os.environ["NGROK_AUTHTOKEN"] = "YOUR_TOKEN"
cfg.ngrok_authtoken = os.environ["NGROK_AUTHTOKEN"]
```

In [ ]:
nest_asyncio.apply()

cfg.ngrok_authtoken = os.environ.get("NGROK_AUTHTOKEN", cfg.ngrok_authtoken)
if cfg.ngrok_authtoken:
    ngrok.set_auth_token(cfg.ngrok_authtoken)
else:
    print("WARNING: NGROK_AUTHTOKEN is empty. Set os.environ['NGROK_AUTHTOKEN'] before this cell if ngrok requires auth.")

# Close old tunnels from previous runs in the same Kaggle kernel.
try:
    ngrok.kill()
except Exception as exc:
    print("ngrok.kill warning:", exc)

def _run_uvicorn():
    uvicorn.run(app, host=cfg.api_host, port=cfg.api_port, log_level="info")

server_thread = threading.Thread(target=_run_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)

public_url = ngrok.connect(cfg.api_port, "http").public_url
predict_url = public_url.rstrip("/") + "/predict"
print("Public API URL:", public_url)
print("POST prediction URL:", predict_url)


## 6. Optional smoke test

In [ ]:
# Optional smoke test after the server starts.
# Replace sample_payload with one official competition query.
import requests

sample_payload = {
    "query_id": "demo-1",
    "type": "type1",
    "query": "Are all drones functioning correctly?",
    "premises": ["All drones function correctly."],
    "options": [],
}

response = requests.post(predict_url, json=sample_payload, timeout=300)
print(response.status_code)
print(response.json())
